In [2]:
import torch

# 1. Dtypes

Dtypes in the PyTorch defines how much space it takes up, what the data represents, and what hardware the operation will run on

1. The first information it's their type:
- Int
- Float
- other
This information indicates if a number have decimal places, or is it an integer or other
2. The number in the last indicates how many bits that numbers allocate in their memory
- 8 (8 bits)
- 16 (16 bits)
- 32 (32 bits)

## Bits vs Bytes 

For understand better the dtypes in the computer science and machine learning, we need to understand bits and bytes

**Bits**: 0 or 1

**Bytes**: a set of bits (normally 8 bits)

and combining both we can write anything with 0s and 1s (to see more https://www.geeksforgeeks.org/computer-organization-architecture/difference-between-bit-and-byte/)

## IEEE 754

We use 3 terms to create a number

- Signal: Normally uses 1 bit, can be - or +
- Expoent: Determine where the comma goes
- Mantissa: Determine the complexity of the number

---

## 1.1 Ints

### The Core Idea

In mathematics, integers are abstract and infinite. In hardware, an integer is a **fixed-size sequence of bits** stored in physical memory. The number of bits determines how large a number you can represent.

### How Bits Translate to Values

Each bit holds exactly **2 possible states**: `0` or `1`. Combining multiple bits multiplies the possibilities:

| Bits | Combinations | Range (unsigned) |
|------|-------------|-----------------|
| 1    | 2¹ = 2      | 0 to 1          |
| 8    | 2⁸ = 256    | 0 to 255        |
| 16   | 2¹⁶ = 65,536 | 0 to 65,535    |
| 32   | 2³² ≈ 4 billion | 0 to 4,294,967,295 |
| 64   | 2⁶⁴ ≈ 18 quintillion | 0 to 18,446,744,073,709,551,615 |

Every additional bit **doubles** the number of representable values.

### Why Different Sizes Exist

A number like `1,000,000,000,000` requires **40 binary digits** to write out. You can verify this in Python:

```python
n = 1_000_000_000_000
print(n.bit_length())  # → 40
```

Since it needs 40 bits, it overflows `int32` (max 32 bits) and must be stored as `int64`.

### Signed vs Unsigned

When negative numbers are needed, **1 bit is reserved for the sign**, leaving the rest for the magnitude:

| Type   | Bits | Range |
|--------|------|-------|
| uint8  | 8    | 0 to 255 |
| int8   | 8    | -128 to 127 |
| uint32 | 32   | 0 to ~4.2 billion |
| int32  | 32   | ~-2.1 billion to ~2.1 billion |

The total number of combinations stays the same — it's just shifted to include negatives.


For uses in python `torch.int`

### Why This Matters in ML

Quantization exploits this directly. Replacing `float32` weights with `int8` reduces memory by **4×** and speeds up inference, at the cost of some precision. Choosing the right bit width is a deliberate trade-off between range, precision, memory, and compute.

In [ ]:
int_8 = torch.tensor([1], dtype = torch.int8)
int_16 = torch.tensor([1000], dtype = torch.int16)
int_32 = torch.tensor([1000000], dtype= torch.int32)
int_64 = torch.tensor([100000000000000], dtype= torch.int64)
# gonna be an error invalid_int = torch.tensor([10000], dtype = torch.int8)


print(f'Int8: {int_8.dtype}')
print(f'Int16: {int_16.dtype}')
print(f'Int32: {int_32.dtype}')
print(f'Int64: {int_64.dtype}')

# For here we only gonna use the
# 1. Signal (+ or -)
# 2. Mantissa (the complexity of the number)

Int8: torch.int8
Int16: torch.int16
Int32: torch.int32
Int64: torch.int64


## 1.2 Floats

### The Core Idea

In mathematics, real numbers are continuous and infinite. In hardware, a float is a **fixed-size sequence of bits** that approximates a real number using scientific notation in base 2. Instead of storing a single integer value, it stores three separate fields: sign, exponent, and mantissa.

### How the Bits Are Split

A `float32` uses 32 bits divided into three fields:

| Field    | Bits | Role                                      |
|----------|------|-------------------------------------------|
| Sign     | 1    | `0` = positive, `1` = negative            |
| Exponent | 8    | Which power of 2 (scale / magnitude)      |
| Mantissa | 23   | Fractional digits within that scale       |

The value is computed as:

```
value = (−1)^sign × 2^(exponent − 127) × (1.mantissa)
```

The `127` is the **bias** — a fixed offset so the exponent can represent both positive and negative powers without a separate sign bit of its own.

### Different Float Types

Not all floats split the bits the same way. The trade-off is always **range vs. precision**:

| Type      | Total bits | Sign | Exponent | Mantissa | Decimal digits |
|-----------|-----------|------|----------|----------|----------------|
| float16   | 16         | 1    | 5        | 10       | ~3             |
| bfloat16  | 16         | 1    | 8        | 7        | ~2–3           |
| float32   | 32         | 1    | 8        | 23       | ~7             |
| float64   | 64         | 1    | 11       | 52       | ~15            |

`bfloat16` keeps the same 8-bit exponent as `float32` (same range), but sacrifices mantissa bits for precision. This is why it is preferred in training: it avoids overflow without needing `float32`.

### Exponent vs. Mantissa — The Intuition

The **exponent** controls *which region* of the number line you are in. Every increment doubles the scale:

```
E = −3  →  interval [0.125, 0.25)
E =  0  →  interval [1.0,   2.0)
E = 10  →  interval [1024,  2048)
```

The **mantissa** controls *how many distinct values* exist inside that interval. With `n` mantissa bits there are `2^n` evenly spaced steps:

```
float32  →  2^23 ≈ 8 million steps per interval  →  gap ≈ 1.2 × 10⁻⁷
float16  →  2^10 = 1024 steps per interval        →  gap ≈ 0.001
```

The key consequence: **relative error is constant, not absolute**. Float gives you the same percentage precision at `0.0001` as at `1,000,000`. This is fundamentally different from fixed-point integers.



| Dtype     | Memory (per param) | Typical use                        |
|-----------|--------------------|------------------------------------|
| float64   | 8 bytes            | Scientific computing, rarely ML    |
| float32   | 4 bytes            | Training reference, optimizer state |
| float16   | 2 bytes            | Training / inference (older GPU)   |


In [ ]:
import torch

float_16 = torch.tensor([1], dtype=torch.float16)
float_32 = torch.tensor([1], dtype=torch.float32)
float_64 = torch.tensor([1], dtype=torch.float64)


print(f'Float16: {float_16.dtype}')
print(f'Float32: {float_32.dtype}')
print(f'Float64: {float_64.dtype}')

# For here we only gonna use the
# 1. Signal (+ or -)
# 2. Expoent (wheres the comma goes)
# 3. Mantissa (the complexity of the number)

Float16: torch.float16
Float32: torch.float32
Float64: torch.float64


## 1.3 BFloats
 
### The Core Idea
 
`bfloat16` ("Brain Float 16") is a 16-bit floating-point format created by Google for TPU training. It is not a standard IEEE 754 type — it is a deliberate truncation of `float32`, keeping the upper 16 bits and discarding the lower 16.
 
The central design decision: **sacrifice precision, preserve range**.
 
### How It Compares to float16 and float32
 
All three use the same sign + exponent + mantissa structure, but split the 16 bits differently:
 
| Type      | Sign | Exponent | Mantissa | Max value      | Decimal digits |
|-----------|------|----------|----------|----------------|----------------|
| float32   | 1    | 8        | 23       | ~3.4 × 10³⁸   | ~7             |
| bfloat16  | 1    | 8        | 7        | ~3.4 × 10³⁸   | ~2–3           |
| float16   | 1    | 5        | 10       | 65504          | ~3             |
 
`bfloat16` and `float32` share the same 8-bit exponent, so they cover **identical ranges**. `float16` uses only 5 exponent bits, capping out at 65504 — small enough to overflow during training.
 

In [6]:
bfloat_16 = torch.tensor([1], dtype=torch.bfloat16)

print(f'BFloat16: {bfloat_16.dtype}')

# For here we only gonna use the
# 1. Signal (+ or -)
# 2. Expoent (wheres the comma goes)
# 3. Mantissa (the complexity of the number)

BFloat16: torch.bfloat16
